# 04 RQ3 Uncertainty-Aware Expert Decision Support Across All Backbone Models

This notebook evaluates whether a risk-aware expert decision-support layer driven by
Monte Carlo Dropout uncertainty, calibrated confidence, and predictive entropy can
reduce unsafe automated decisions under real-world conditions.

It performs:
- checkpoint loading
- Monte Carlo Dropout inference
- temperature scaling
- predictive entropy calculation
- expert-rule decision support
- safety comparison against a vision-only baseline
- export of tables and figures

Outputs:
- Table 5: risk-aware decision support comparison
- Table 6: calibration and uncertainty summary
- Figure 5: reliability of confidence estimates before and after calibration
- Figure 6: comparative safety of vision-only and uncertainty-aware decision support
- ZIP archive of RQ3 outputs

In [1]:
# ----------------------------------------
# Section 1: Imports
# ----------------------------------------

import os
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.calibration import calibration_curve

In [2]:
# ----------------------------------------
# Section 2: Reproducibility setup
# ----------------------------------------

SEED = 42

def seed_everything(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    """
    Ensure each DataLoader worker uses a deterministic seed.
    """
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

print("Reproducibility setup completed")
print(f"Global seed: {SEED}")

Reproducibility setup completed
Global seed: 42


In [3]:
# ----------------------------------------
# Section 3: Configuration
# ----------------------------------------

CONFIG = {
    "seed": SEED,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,
    "mc_passes": 30,
    "plantvillage_root": "/kaggle/input/datasets/thedataeng/plantvillage",
    "plantdoc_root": "/kaggle/input/datasets/thedataeng/plantdoc",

    # Update this if your Kaggle dataset input name differs
    "checkpoint_root": "/kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs",

    "model_names": ["resnet50", "efficientnet_b0", "mobilenet_v2"],
    "output_root": "/kaggle/working/thesis_outputs/rq3_uncertainty_expert_system",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = Path(CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Configuration loaded")
print(f"Device: {DEVICE}")
print(f"Models: {CONFIG['model_names']}")
print(f"MC Dropout passes: {CONFIG['mc_passes']}")
print(f"Output root: {OUTPUT_ROOT}")

Configuration loaded
Device: cuda
Models: ['resnet50', 'efficientnet_b0', 'mobilenet_v2']
MC Dropout passes: 30
Output root: /kaggle/working/thesis_outputs/rq3_uncertainty_expert_system


In [5]:
# ----------------------------------------
# Section 4: Helper functions
# ----------------------------------------

def ensure_dir(path: Path) -> Path:
    """
    Create a directory if it does not exist and return the Path object.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path

def get_eval_transform(image_size: int = 224):
    """
    Create the evaluation transform used across RQ3 experiments.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

def create_model(model_name: str, num_classes: int) -> nn.Module:
    """
    Create the selected backbone model and replace the classification head.
    """
    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    return model.to(DEVICE)

def checkpoint_path(model_name: str) -> Path:
    """
    Return the checkpoint path for the selected model.
    """
    return Path(CONFIG["checkpoint_root"]) / "checkpoints" / f"{model_name}_seed{SEED}_best.pt"

def enable_dropout(model: nn.Module) -> None:
    """
    Enable dropout layers during inference for Monte Carlo Dropout.
    """
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()

@torch.no_grad()
def collect_logits_labels(model: nn.Module, loader: DataLoader):
    """
    Collect deterministic logits and labels from a loader.
    Used for calibration and baseline comparison.
    """
    model.eval()

    all_logits = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)
        logits = model(images)

        all_logits.append(logits.cpu())
        all_labels.append(labels)

    return torch.cat(all_logits, dim=0), torch.cat(all_labels, dim=0)

@torch.no_grad()
def mc_dropout_predict(model: nn.Module, loader: DataLoader, n_passes: int = 30):
    """
    Perform Monte Carlo Dropout inference across a DataLoader and return:
    - mean probabilities
    - predictive entropy
    - predicted class labels
    - true labels
    """
    model.eval()
    enable_dropout(model)

    all_mean_probs = []
    all_entropy = []
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)

        pass_probs = []

        for _ in range(n_passes):
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            pass_probs.append(probs.unsqueeze(0))

        pass_probs = torch.cat(pass_probs, dim=0)
        mean_probs = pass_probs.mean(dim=0)

        entropy = -(mean_probs * torch.log(mean_probs.clamp(min=1e-12))).sum(dim=1)
        preds = mean_probs.argmax(dim=1)

        all_mean_probs.append(mean_probs.cpu().numpy())
        all_entropy.append(entropy.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())

    return {
        "mean_probs": np.vstack(all_mean_probs),
        "entropy": np.concatenate(all_entropy),
        "preds": np.concatenate(all_preds),
        "labels": np.concatenate(all_labels),
    }

def fit_temperature(logits: torch.Tensor, labels: torch.Tensor, max_iter: int = 100) -> float:
    """
    Fit a scalar temperature using validation logits and labels.
    """
    temperature = torch.ones(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=max_iter)
    criterion = nn.CrossEntropyLoss()

    labels = labels.long()

    def closure():
        optimizer.zero_grad()
        loss = criterion(logits / temperature.clamp(min=1e-3), labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(temperature.detach().item())

def ece_score(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    """
    Compute Expected Calibration Error (ECE).
    """
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = predictions == labels

    bin_bounds = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        in_bin = (confidences > bin_bounds[i]) & (confidences <= bin_bounds[i + 1])
        if in_bin.any():
            acc_bin = accuracies[in_bin].mean()
            conf_bin = confidences[in_bin].mean()
            ece += np.abs(acc_bin - conf_bin) * in_bin.mean()

    return float(ece)

def normalize_entropy(entropy: np.ndarray, num_classes: int) -> np.ndarray:
    """
    Normalize entropy to the range [0, 1] using the maximum possible entropy.
    """
    max_entropy = np.log(num_classes + 1e-12)
    return entropy / max_entropy

def expert_decision(conf: float, ent: float) -> tuple:
    """
    Apply the rule-based expert decision-support logic.
    Returns (action, recommendation).
    """
    if conf >= 0.85 and ent <= 0.30:
        return "Accept", "Recommend treatment or monitoring by predicted class"
    elif conf >= 0.60 and ent <= 0.55:
        return "Monitor", "Monitor and retake if symptoms persist"
    elif conf >= 0.45:
        return "Retake", "Request a clearer image"
    return "Review", "Seek expert review"

def save_table(df: pd.DataFrame, name: str) -> None:
    """
    Save a DataFrame as CSV in the tables directory.
    """
    table_dir = ensure_dir(OUTPUT_ROOT / "tables")
    csv_path = table_dir / f"{name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved table: {csv_path}")

def save_figure(fig: plt.Figure, name: str) -> None:
    """
    Save a matplotlib figure as PDF in the figures directory.
    """
    fig_dir = ensure_dir(OUTPUT_ROOT / "figures")
    pdf_path = fig_dir / f"{name}.pdf"
    fig.tight_layout()
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {pdf_path}")

def pretty_metric(x: float) -> float:
    """
    Round a metric value for cleaner table presentation.
    """
    return round(float(x), 4)

print('Done')

Done


In [6]:
# ----------------------------------------
# Section 5: Dataset loading
# ----------------------------------------

pv_root = Path(CONFIG["plantvillage_root"])
pd_root = Path(CONFIG["plantdoc_root"])

val_dir = pv_root / "val"

assert val_dir.exists(), f"Missing PlantVillage validation directory: {val_dir}"
assert pd_root.exists(), f"Missing PlantDoc directory: {pd_root}"

eval_tfms = get_eval_transform(CONFIG["image_size"])

val_dataset = datasets.ImageFolder(val_dir, transform=eval_tfms)
plantdoc_dataset = datasets.ImageFolder(pd_root, transform=eval_tfms)

generator = torch.Generator()
generator.manual_seed(SEED)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

plantdoc_loader = DataLoader(
    plantdoc_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

CLASS_NAMES = plantdoc_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print("Datasets loaded successfully")
print(f"Validation samples: {len(val_dataset)}")
print(f"PlantDoc samples:   {len(plantdoc_dataset)}")
print(f"Number of classes:  {NUM_CLASSES}")

Datasets loaded successfully
Validation samples: 5518
PlantDoc samples:   2555
Number of classes:  27


In [7]:
# ----------------------------------------
# Section 6: Checkpoint loading
# ----------------------------------------

models_loaded = {}

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Loading checkpoint {idx}/{len(CONFIG['model_names'])}: {model_name}")

    ckpt_path = checkpoint_path(model_name)
    assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"

    model = create_model(model_name, NUM_CLASSES)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    models_loaded[model_name] = model

    print(f"Loaded checkpoint: {ckpt_path}")

print("All checkpoints loaded successfully")

Loading checkpoint 1/3: resnet50
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/resnet50_seed42_best.pt
Loading checkpoint 2/3: efficientnet_b0
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/efficientnet_b0_seed42_best.pt
Loading checkpoint 3/3: mobilenet_v2
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/mobilenet_v2_seed42_best.pt
All checkpoints loaded successfully


In [8]:
# ----------------------------------------
# Section 7: Temperature scaling for all models
# ----------------------------------------

temperature_map = {}

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Fitting temperature for model {idx}/{len(CONFIG['model_names'])}: {model_name}")

    model = models_loaded[model_name]

    print("  Collecting validation logits")
    val_logits, val_labels = collect_logits_labels(model, val_loader)

    print("  Fitting temperature parameter")
    temperature = fit_temperature(val_logits, val_labels)

    temperature_map[model_name] = temperature

    print(f"  Learned temperature: {temperature:.4f}")

print("Temperature scaling completed for all models")

Fitting temperature for model 1/3: resnet50
  Fitting temperature parameter
  Learned temperature: 1.0020
Fitting temperature for model 2/3: efficientnet_b0
  Fitting temperature parameter
  Learned temperature: 1.1691
Fitting temperature for model 3/3: mobilenet_v2
  Fitting temperature parameter
  Learned temperature: 0.9974
Temperature scaling completed for all models


In [9]:
# ----------------------------------------
# Section 8: Deterministic prediction and calibration summary for all models
# ----------------------------------------

deterministic_results = {}

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Preparing deterministic PlantDoc predictions for model {idx}/{len(CONFIG['model_names'])}: {model_name}")

    model = models_loaded[model_name]
    temperature = temperature_map[model_name]

    pd_logits, pd_labels_t = collect_logits_labels(model, plantdoc_loader)
    pd_labels = pd_labels_t.numpy()

    probs_before = torch.softmax(pd_logits, dim=1).numpy()
    probs_after = torch.softmax(pd_logits / temperature, dim=1).numpy()

    pred_before = probs_before.argmax(axis=1)
    pred_after = probs_after.argmax(axis=1)

    ece_before = ece_score(probs_before, pd_labels)
    ece_after = ece_score(probs_after, pd_labels)

    brier_before = np.mean(np.sum((probs_before - np.eye(NUM_CLASSES)[pd_labels]) ** 2, axis=1))
    brier_after = np.mean(np.sum((probs_after - np.eye(NUM_CLASSES)[pd_labels]) ** 2, axis=1))

    high_conf_mis_before = float(((pred_before != pd_labels) & (probs_before.max(axis=1) >= 0.80)).mean() * 100)
    high_conf_mis_after = float(((pred_after != pd_labels) & (probs_after.max(axis=1) >= 0.80)).mean() * 100)

    deterministic_results[model_name] = {
        "pd_labels": pd_labels,
        "probs_before": probs_before,
        "probs_after": probs_after,
        "pred_before": pred_before,
        "pred_after": pred_after,
        "ece_before": ece_before,
        "ece_after": ece_after,
        "brier_before": brier_before,
        "brier_after": brier_after,
        "high_conf_mis_before": high_conf_mis_before,
        "high_conf_mis_after": high_conf_mis_after,
    }

    print(f"  ECE before: {ece_before:.4f}")
    print(f"  ECE after:  {ece_after:.4f}")

print("Deterministic prediction and calibration summaries completed for all models")

Preparing deterministic PlantDoc predictions for model 1/3: resnet50
  ECE before: 0.5404
  ECE after:  0.5399
Preparing deterministic PlantDoc predictions for model 2/3: efficientnet_b0
  ECE before: 0.5558
  ECE after:  0.5070
Preparing deterministic PlantDoc predictions for model 3/3: mobilenet_v2
  ECE before: 0.3719
  ECE after:  0.3728
Deterministic prediction and calibration summaries completed for all models


In [10]:
# ----------------------------------------
# Section 9: Monte Carlo Dropout inference for all models
# ----------------------------------------

mc_results_map = {}

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Running MC Dropout inference for model {idx}/{len(CONFIG['model_names'])}: {model_name}")

    model = models_loaded[model_name]

    mc_results = mc_dropout_predict(
        model=model,
        loader=plantdoc_loader,
        n_passes=CONFIG["mc_passes"]
    )

    entropy_norm = normalize_entropy(mc_results["entropy"], NUM_CLASSES)

    mc_results_map[model_name] = {
        "mean_probs": mc_results["mean_probs"],
        "entropy": mc_results["entropy"],
        "norm_entropy": entropy_norm,
        "preds": mc_results["preds"],
        "labels": mc_results["labels"],
    }

print("MC Dropout inference completed successfully for all models")

Running MC Dropout inference for model 1/3: resnet50
Running MC Dropout inference for model 2/3: efficientnet_b0
Running MC Dropout inference for model 3/3: mobilenet_v2
MC Dropout inference completed successfully for all models


In [11]:
# ----------------------------------------
# Section 10: Expert decision-support evaluation for all models
# ----------------------------------------

table5_rows = []
table6_rows = []
decision_logs = {}

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Applying expert decision-support layer for model {idx}/{len(CONFIG['model_names'])}: {model_name}")

    det = deterministic_results[model_name]
    mc = mc_results_map[model_name]

    pd_labels = det["pd_labels"]
    pred_before = det["pred_before"]
    pred_after = det["pred_after"]
    probs_before = det["probs_before"]
    probs_after = det["probs_after"]

    calibrated_conf = probs_after.max(axis=1)
    norm_entropy = mc["norm_entropy"]

    decision_rows = []

    for i in range(len(pd_labels)):
        conf = float(calibrated_conf[i])
        ent = float(norm_entropy[i])

        action, recommendation = expert_decision(conf, ent)

        decision_rows.append({
            "TrueClass": CLASS_NAMES[pd_labels[i]],
            "PredictedClass": CLASS_NAMES[pred_after[i]],
            "Confidence": pretty_metric(conf),
            "NormEntropy": pretty_metric(ent),
            "Action": action,
            "Recommendation": recommendation,
            "Correct": int(pred_after[i] == pd_labels[i]),
        })

    decisions_df = pd.DataFrame(decision_rows)
    decision_logs[model_name] = decisions_df

    unsafe_accepts_cnn = int(((pred_before != pd_labels) & (probs_before.max(axis=1) >= 0.60)).sum())

    expert_accept_mask = decisions_df["Action"].values == "Accept"
    expert_defer_mask = np.isin(decisions_df["Action"].values, ["Retake", "Review"])
    expert_correct_mask = decisions_df["Correct"].values == 1

    unsafe_accepts_expert = int((expert_accept_mask & (~expert_correct_mask)).sum())
    deferred_cases = int(expert_defer_mask.sum())
    correct_escalations = int((expert_defer_mask & (~expert_correct_mask)).sum())

    decision_safety_cnn = 1 - unsafe_accepts_cnn / len(decisions_df)
    decision_safety_expert = 1 - unsafe_accepts_expert / len(decisions_df)

    table5_rows.append({
        "Model": model_name,
        "System Variant": "CNN only",
        "Unsafe Accepts": unsafe_accepts_cnn,
        "Deferred Cases": 0,
        "Correct Escalations": 0,
        "Decision Safety": pretty_metric(decision_safety_cnn),
    })

    table5_rows.append({
        "Model": model_name,
        "System Variant": "CNN + uncertainty-aware expert layer",
        "Unsafe Accepts": unsafe_accepts_expert,
        "Deferred Cases": deferred_cases,
        "Correct Escalations": correct_escalations,
        "Decision Safety": pretty_metric(decision_safety_expert),
    })

    table6_rows.append({
        "Model": model_name,
        "ECE Before": pretty_metric(det["ece_before"]),
        "ECE After": pretty_metric(det["ece_after"]),
        "Brier Score Before": pretty_metric(det["brier_before"]),
        "Brier Score After": pretty_metric(det["brier_after"]),
        "Avg. Predictive Entropy": pretty_metric(norm_entropy.mean()),
        "High-Confidence Misclassifications Before (%)": pretty_metric(det["high_conf_mis_before"]),
        "High-Confidence Misclassifications After (%)": pretty_metric(det["high_conf_mis_after"]),
    })

print("Expert decision-support evaluation completed successfully for all models")

Applying expert decision-support layer for model 1/3: resnet50
Applying expert decision-support layer for model 2/3: efficientnet_b0
Applying expert decision-support layer for model 3/3: mobilenet_v2
Expert decision-support evaluation completed successfully for all models


In [12]:
# ----------------------------------------
# Section 11: Save Table 5 - Safety comparison
# ----------------------------------------

table5_df = pd.DataFrame(table5_rows)

pretty_names = {
    "resnet50": "ResNet50",
    "efficientnet_b0": "EfficientNet-B0",
    "mobilenet_v2": "MobileNetV2",
}
table5_df["Model"] = table5_df["Model"].map(pretty_names)

save_table(table5_df, "Table_5_RiskAware_Decision_Support_Comparison")

print("Table 5 saved successfully")
display(table5_df)

Saved table: /kaggle/working/thesis_outputs/rq3_uncertainty_expert_system/tables/Table_5_RiskAware_Decision_Support_Comparison.csv
Table 5 saved successfully


,Model,System Variant,Unsafe Accepts,Deferred Cases,Correct Escalations,Decision Safety
0,ResNet50,CNN only,1369,0,0,0.4642
1,ResNet50,CNN + uncertainty-aware expert layer,831,772,672,0.6748
2,EfficientNet-B0,CNN only,1368,0,0,0.4646
3,EfficientNet-B0,CNN + uncertainty-aware expert layer,579,1004,904,0.7734
4,MobileNetV2,CNN only,793,0,0,0.6896
5,MobileNetV2,CNN + uncertainty-aware expert layer,341,1380,1194,0.8665


In [13]:
# ----------------------------------------
# Section 12: Save Table 6 - Calibration and uncertainty summary
# ----------------------------------------
pretty_names = {
    "resnet50": "ResNet50",
    "efficientnet_b0": "EfficientNet-B0",
    "mobilenet_v2": "MobileNetV2",
}

table6_df = pd.DataFrame(table6_rows)
table6_df["Model"] = table6_df["Model"].map(pretty_names)

save_table(table6_df, "Table_6_Calibration_and_Uncertainty_Summary")

print("Table 6 saved successfully")
display(table6_df)

Saved table: /kaggle/working/thesis_outputs/rq3_uncertainty_expert_system/tables/Table_6_Calibration_and_Uncertainty_Summary.csv
Table 6 saved successfully


,Model,ECE Before,ECE After,Brier Score Before,Brier Score After,Avg. Predictive Entropy,High-Confidence Misclassifications Before (%),High-Confidence Misclassifications After (%)
0,ResNet50,0.5404,0.5399,1.2664,1.2658,0.2418,37.1820,37.0646
1,EfficientNet-B0,0.5558,0.5070,1.2858,1.2293,0.2587,34.5597,28.1409
2,MobileNetV2,0.3719,0.3728,1.0760,1.0769,0.4178,16.9863,17.0646


In [14]:
# ----------------------------------------
# Section 13: Save sample decision logs
# ----------------------------------------

for model_name, decisions_df in decision_logs.items():
    pretty_model_name = pretty_names[model_name]
    sample_decision_df = decisions_df.head(100).copy()
    save_table(sample_decision_df, f"Sample_Decision_Log_First_100_{pretty_model_name}")

print("Sample decision logs saved successfully")

Saved table: /kaggle/working/thesis_outputs/rq3_uncertainty_expert_system/tables/Sample_Decision_Log_First_100_ResNet50.csv
Saved table: /kaggle/working/thesis_outputs/rq3_uncertainty_expert_system/tables/Sample_Decision_Log_First_100_EfficientNet-B0.csv
Saved table: /kaggle/working/thesis_outputs/rq3_uncertainty_expert_system/tables/Sample_Decision_Log_First_100_MobileNetV2.csv
Sample decision logs saved successfully


In [20]:
# ----------------------------------------
# Section 14: Figure 5 - Reliability diagram
# ----------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharex=True, sharey=True)

model_order = ["resnet50", "efficientnet_b0", "mobilenet_v2"]

for ax, model_name in zip(axes, model_order):
    det = deterministic_results[model_name]
    pretty_model = pretty_names[model_name]

    pd_labels = det["pd_labels"]

    conf_before = det["probs_before"].max(axis=1)
    pred_before = det["pred_before"]
    frac_pos_b, mean_pred_b = calibration_curve(
        (pred_before == pd_labels).astype(int),
        conf_before,
        n_bins=10,
        strategy="uniform"
    )

    conf_after = det["probs_after"].max(axis=1)
    pred_after = det["pred_after"]
    frac_pos_a, mean_pred_a = calibration_curve(
        (pred_after == pd_labels).astype(int),
        conf_after,
        n_bins=10,
        strategy="uniform"
    )

    ax.plot([0, 1], [0, 1], "--", linewidth=1.5, label="Ideal calibration")
    ax.plot(mean_pred_b, frac_pos_b, marker="o", linewidth=1.8, label="Before")
    ax.plot(mean_pred_a, frac_pos_a, marker="s", linewidth=1.8, label="After")

    ax.set_title(pretty_model)
    ax.set_xlabel("Predicted confidence")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Observed accuracy")
axes[0].legend(frameon=True, fontsize=8)
fig.suptitle("Figure 5. Reliability of Confidence Estimates Before and After Calibration", y=1.03)

save_figure(fig, "Figure_5_Reliability_Before_After_Calibration_All_Models")

Saved figure: /kaggle/working/thesis_outputs/rq3_uncertainty_expert_system/figures/Figure_5_Reliability_Before_After_Calibration_All_Models.pdf


In [16]:
# ----------------------------------------
# Section 15: Figure 6 - Comparative safety profile
# ----------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 5.5))

plot_df = table5_df.pivot(
    index="Model",
    columns="System Variant",
    values="Unsafe Accepts"
).reset_index()

model_order = ["ResNet50", "EfficientNet-B0", "MobileNetV2"]
plot_df = plot_df.set_index("Model").loc[model_order].reset_index()

x = np.arange(len(plot_df))
width = 0.35

ax.bar(
    x - width / 2,
    plot_df["CNN only"],
    width=width,
    label="CNN only"
)

ax.bar(
    x + width / 2,
    plot_df["CNN + uncertainty-aware expert layer"],
    width=width,
    label="CNN + uncertainty-aware expert layer"
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["Model"])
ax.set_ylabel("Unsafe accepted predictions")
ax.set_title("Figure 6. Comparative Safety Profile of Vision-Only and Uncertainty-Aware Decision Support")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=True)

save_figure(fig, "Figure_6_Comparative_Safety_Profile_All_Models")

Saved figure: /kaggle/working/thesis_outputs/rq3_uncertainty_expert_system/figures/Figure_6_Comparative_Safety_Profile_All_Models.pdf


In [17]:
# ----------------------------------------
# Section 16: Save RQ3 metadata
# ----------------------------------------

meta_dir = ensure_dir(OUTPUT_ROOT / "metadata")

rq3_meta = {
    "seed": SEED,
    "models_evaluated": CONFIG["model_names"],
    "mc_passes": CONFIG["mc_passes"],
    "temperatures": {k: pretty_metric(v) for k, v in temperature_map.items()},
    "num_classes": NUM_CLASSES,
    "num_samples": len(next(iter(decision_logs.values()))),
}

meta_path = meta_dir / "rq3_metadata.json"
with open(meta_path, "w") as f:
    json.dump(rq3_meta, f, indent=2)

print("RQ3 metadata saved successfully")
print(f"Metadata path: {meta_path}")

RQ3 metadata saved successfully
Metadata path: /kaggle/working/thesis_outputs/rq3_uncertainty_expert_system/metadata/rq3_metadata.json


In [18]:
# ----------------------------------------
# Section 17: Create ZIP archive
# ----------------------------------------

zip_path = OUTPUT_ROOT.parent / "04_rq3_uncertainty_expert_system_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("ZIP archive created successfully")
print(f"ZIP file: {zip_path}")
print("04_rq3_uncertainty_expert_system notebook completed successfully")

ZIP archive created successfully
ZIP file: /kaggle/working/thesis_outputs/04_rq3_uncertainty_expert_system_outputs.zip
04_rq3_uncertainty_expert_system notebook completed successfully
